# LoRA Fine-Tuning ESM2-650M for Fluorescent Protein Property Prediction

## Overview

**LoRA** (Low-Rank Adaptation) fine-tuning of **ESM2-650M** (`esm2_t33_650M_UR50D`) with
chromophore-aware attention pooling for multi-task prediction of 5 photophysical properties.

**Architecture**: ESM2 (frozen) + LoRA rank-8 adapters (last 6 layers, q/v projections) →
4-head attention pooling with chromophore position bias → 5 MLP prediction heads

**Targets**: ex_max, em_max, qy, ext_coeff, pka (masked loss for missing values)

**Evaluation**: 20-fold CV matching ChromaFormer baselines for direct comparison

**GPU**: A100 40GB recommended — ~1.5h per fold, ~30h total (resume support across sessions)

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q fair-esm scikit-learn matplotlib

In [ ]:
# ── Cell 2: Mount Drive, imports, config ──────────────────────────────────────
import os, gc, time, json, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr
from pathlib import Path
import matplotlib.pyplot as plt
import esm

warnings.filterwarnings("ignore")

# ── Environment detection ─────────────────────────────────────────────────────
IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/Othercomputers/My Mac/FluorCode/data")
    OUT_DIR   = Path("/content/drive/MyDrive/FluorCode/results_lora_esm2")
else:
    # Local execution: edit these paths to match your setup
    REPO_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path("../..")
    DRIVE_DIR = REPO_ROOT / "data"
    OUT_DIR   = REPO_ROOT / "model" / "LoRA_ESM2" / "output"
    print("Running locally. Edit DRIVE_DIR / OUT_DIR if paths don't match your setup.")

OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Hyperparameters ───────────────────────────────────────────────────────────
TARGETS     = ["ex_max", "em_max", "qy", "ext_coeff", "pka"]
N_FOLDS     = 20
MAX_SEQ_LEN = 1024
BATCH_SIZE  = 8
ACCUM_STEPS = 2           # effective batch = 16
MAX_EPOCHS  = 100
PATIENCE    = 10
WARMUP_EPOCHS = 5
LORA_RANK   = 16          # ↑ from 8 (more adaptation capacity)
LORA_ALPHA  = 32          # ↑ from 16 (keep scaling = alpha/rank = 2.0)
LORA_LAYERS = list(range(27, 33))  # last 6 of 33 layers (0-indexed)
LR_LORA     = 5e-4
LR_HEAD     = 1e-4
WEIGHT_DECAY = 1e-2
GRAD_CLIP   = 1.0
DROPOUT     = 0.1
N_ATTN_HEADS = 4
EMBED_DIM   = 1280
RANDOM_SEED = 42

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# ── Cell 3: Load data ─────────────────────────────────────────────────────────
SEQ_DIR = DRIVE_DIR / "sequence" if not IS_COLAB else DRIVE_DIR
meta = pd.read_csv(SEQ_DIR / "fp_cleaned.csv")
chrom = pd.read_csv(SEQ_DIR / "chromophore_positions.csv")

# Merge chromophore positions
meta = meta.merge(
    chrom[["slug", "chrom_pos0", "chrom_pos1", "chrom_pos2", "status"]],
    on="slug", how="left",
)

# Filter: need sequence, within max length
meta = meta[meta["seq"].notna() & (meta["seq"].str.len() > 0)].reset_index(drop=True)
meta = meta[meta["seq"].str.len() <= MAX_SEQ_LEN].reset_index(drop=True)

# Exclude cofactor-dependent FPs (biliverdin, flavin, etc.) — keep GFP family only
n_before = len(meta)
meta = meta[meta["cofactor"].isna()].reset_index(drop=True)
print(f"Excluded {n_before - len(meta)} cofactor-dependent FPs")

print(f"Total FPs (GFP family): {len(meta)}")
for t in TARGETS:
    n = meta[t].notna().sum()
    print(f"  {t:12s}: {n:3d} samples ({n/len(meta)*100:.0f}%)")

n_chrom = (meta["status"] == "ok").sum()
print(f"\nChromophore positions available: {n_chrom}/{len(meta)}")
print(f"Sequence length range: {meta['seq'].str.len().min()} – {meta['seq'].str.len().max()}")

In [ ]:
# ── Cell 4: LoRA module + Load ESM2 ───────────────────────────────────────────
import functools
import torch.utils.checkpoint as grad_ckpt

class LoRALinear(nn.Module):
    """Low-Rank Adaptation wrapper for nn.Linear.

    Computes: output = W_frozen(x) + (alpha/rank) * x @ A @ B
    A: (in_features, rank) — init normal(0, 0.01)
    B: (rank, out_features) — init zeros (so LoRA starts as identity)
    """

    def __init__(self, orig_linear: nn.Linear, rank: int = 8, alpha: float = 16.0):
        super().__init__()
        self.orig_linear = orig_linear
        self.rank = rank
        self.scaling = alpha / rank

        in_f = orig_linear.in_features
        out_f = orig_linear.out_features

        self.lora_A = nn.Parameter(torch.randn(in_f, rank) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(rank, out_f))

        # Freeze original
        orig_linear.weight.requires_grad = False
        if orig_linear.bias is not None:
            orig_linear.bias.requires_grad = False

    def forward(self, x):
        base_out = self.orig_linear(x)
        lora_out = self.scaling * (x @ self.lora_A @ self.lora_B)
        return base_out + lora_out


def apply_lora(model, target_layers, rank=8, alpha=16.0):
    """Inject LoRA into q_proj, k_proj, v_proj, out_proj of specified ESM2 layers."""
    n_params = 0
    for layer_idx in target_layers:
        layer = model.layers[layer_idx]
        attn = layer.self_attn

        for proj_name in ["q_proj", "k_proj", "v_proj", "out_proj"]:
            orig = getattr(attn, proj_name)
            setattr(attn, proj_name, LoRALinear(orig, rank=rank, alpha=alpha))
            n_params += rank * (orig.in_features + orig.out_features)

    return n_params


def enable_gradient_checkpointing(esm_model):
    """Wrap each ESM2 transformer layer forward with gradient checkpointing."""
    for layer in esm_model.layers:
        orig = layer.forward

        def make_ckpt_fwd(orig_fwd):
            @functools.wraps(orig_fwd)
            def ckpt_fwd(*args, **kwargs):
                if not torch.is_grad_enabled():
                    return orig_fwd(*args, **kwargs)
                def fn(*a):
                    return orig_fwd(*a, **kwargs)
                return grad_ckpt.checkpoint(fn, *args, use_reentrant=False)
            return ckpt_fwd

        layer.forward = make_ckpt_fwd(orig)

    print(f"Gradient checkpointing: wrapped {len(esm_model.layers)} transformer layers")


# ── Load ESM2-650M ────────────────────────────────────────────────────────────
print("Loading ESM2-650M (esm2_t33_650M_UR50D)...")
esm_model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
batch_converter = alphabet.get_batch_converter()

# Freeze all base params first
for p in esm_model.parameters():
    p.requires_grad = False

# Inject LoRA adapters (now q/k/v/out_proj)
n_lora = apply_lora(esm_model, LORA_LAYERS, rank=LORA_RANK, alpha=LORA_ALPHA)

# Enable gradient checkpointing to save VRAM
enable_gradient_checkpointing(esm_model)

esm_model = esm_model.to(DEVICE)

total_params = sum(p.numel() for p in esm_model.parameters())
lora_params  = sum(p.numel() for p in esm_model.parameters() if p.requires_grad)

print(f"ESM2 total params:     {total_params / 1e6:.0f}M")
print(f"LoRA trainable params: {lora_params:,} ({lora_params / total_params * 100:.3f}%)")
print(f"LoRA config: rank={LORA_RANK}, alpha={LORA_ALPHA}, layers={LORA_LAYERS}")
print(f"LoRA targets: q_proj, k_proj, v_proj, out_proj")

gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
print("ESM2 + LoRA ready.")

In [ ]:
# ── Cell 5: Model architecture ────────────────────────────────────────────────

TASK_WEIGHTS = {"ex_max": 1.0, "em_max": 1.0, "qy": 1.0, "ext_coeff": 0.3, "pka": 0.3}

class ChromophoreAwareAttentionPooling(nn.Module):
    """Multi-head attention pooling with learned chromophore position bias.

    4 independent attention heads compute per-residue scores. A learnable scalar
    bias is added at chromophore triad positions to encourage the model to attend
    to the chromophore region. Heads are concatenated and projected back to D.
    """

    def __init__(self, hidden_dim=1280, n_heads=4):
        super().__init__()
        self.n_heads = n_heads

        self.heads = nn.ModuleList([
            nn.Linear(hidden_dim, 1, bias=True) for _ in range(n_heads)
        ])
        self.chrom_bias = nn.Parameter(torch.tensor(3.0))  # ↑ from 1.0
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim * n_heads, hidden_dim // 2),  # 5120 → 640
            nn.GELU(),
            nn.LayerNorm(hidden_dim // 2),
            nn.Linear(hidden_dim // 2, hidden_dim),            # 640 → 1280
        )

    def forward(self, hidden_states, seq_lens, chrom_positions=None):
        """
        Args:
            hidden_states: (B, L, D) — ESM2 output (includes BOS/EOS tokens)
            seq_lens: (B,) — raw sequence lengths (without BOS/EOS)
            chrom_positions: (B, 3) — 0-indexed positions, -1 if unavailable
        Returns:
            pooled: (B, D)
            attn_weights: (B, L) — averaged across heads
        """
        B, L, D = hidden_states.shape

        # Mask: residues sit at positions 1..seq_len (BOS=0, EOS=seq_len+1)
        mask = torch.zeros(B, L, device=hidden_states.device)
        for i in range(B):
            mask[i, 1 : seq_lens[i] + 1] = 1.0

        # Chromophore mask (positions offset by +1 for BOS)
        chrom_mask = torch.zeros(B, L, device=hidden_states.device)
        if chrom_positions is not None:
            for i in range(B):
                for p in chrom_positions[i]:
                    if 0 <= p < seq_lens[i]:
                        chrom_mask[i, p + 1] = 1.0

        head_outputs = []
        all_weights = []

        for head in self.heads:
            scores = head(hidden_states).squeeze(-1)        # (B, L)
            scores = scores + self.chrom_bias * chrom_mask   # chromophore bias
            scores = scores.masked_fill(mask == 0, -1e9)
            weights = torch.softmax(scores, dim=-1)          # (B, L)

            pooled_head = (hidden_states * weights.unsqueeze(-1)).sum(dim=1)
            head_outputs.append(pooled_head)
            all_weights.append(weights)

        pooled = self.proj(torch.cat(head_outputs, dim=-1))  # (B, D)
        avg_weights = torch.stack(all_weights).mean(dim=0)   # (B, L)
        return pooled, avg_weights


class LoRAESM2MultiTask(nn.Module):
    """ESM2 + LoRA backbone → chromophore-aware pooling → 5 task heads."""

    def __init__(self, esm_backbone, n_heads=4, dropout=0.1):
        super().__init__()
        self.esm = esm_backbone
        self.pool = ChromophoreAwareAttentionPooling(EMBED_DIM, n_heads=n_heads)

        self.heads = nn.ModuleDict({
            t: nn.Sequential(
                nn.LayerNorm(EMBED_DIM),
                nn.Dropout(dropout),
                nn.Linear(EMBED_DIM, 256),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(256, 1),
            )
            for t in TARGETS
        })

    def forward(self, tokens, seq_lens, chrom_positions=None):
        # ESM2 forward — last layer representation
        results = self.esm(tokens, repr_layers=[33], return_contacts=False)
        hidden = results["representations"][33]  # (B, L, 1280)

        # Pool
        pooled, attn_wts = self.pool(hidden, seq_lens, chrom_positions)

        # Predict each target
        preds = {t: head(pooled).squeeze(-1) for t, head in self.heads.items()}
        return preds, attn_wts, pooled


def masked_huber_loss(preds, targets, delta=1.0):
    """Weighted Huber loss averaged over non-NaN targets across all tasks."""
    total_loss = torch.tensor(0.0, device=DEVICE)
    total_weight = 0.0

    for name in preds:
        valid = ~torch.isnan(targets[name])
        if valid.sum() == 0:
            continue
        loss = nn.functional.huber_loss(
            preds[name][valid], targets[name][valid],
            reduction="mean", delta=delta,
        )
        w = TASK_WEIGHTS.get(name, 1.0)
        total_loss = total_loss + w * loss
        total_weight += w

    return total_loss / max(total_weight, 1e-8)


# ── Build model ───────────────────────────────────────────────────────────────
model = LoRAESM2MultiTask(esm_model, n_heads=N_ATTN_HEADS, dropout=DROPOUT).to(DEVICE)

pool_params = sum(p.numel() for p in model.pool.parameters())
head_params = sum(p.numel() for p in model.heads.parameters())
all_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model summary:")
print(f"  LoRA params:     {lora_params:>10,}")
print(f"  Pool params:     {pool_params:>10,}")
print(f"  Head params:     {head_params:>10,}")
print(f"  Total trainable: {all_trainable:>10,}")
print(f"  Frozen ESM2:     {total_params - lora_params:>10,}")
print(f"  Task weights:    {TASK_WEIGHTS}")

In [ ]:
# ── Cell 6: Dataset & DataLoader ─────────────���────────────────────────────────

class FPDataset(Dataset):
    """Fluorescent protein dataset for ESM2 multi-task LoRA training."""

    def __init__(self, df, alphabet, target_stats=None):
        """
        Args:
            df: DataFrame with slug, seq, target cols, chromophore position cols
            alphabet: ESM2 alphabet for tokenization
            target_stats: dict[target -> (mean, std)] — if None, computed from df
        """
        self.slugs = df["slug"].tolist()
        self.seqs = df["seq"].tolist()
        self.seq_lens = [len(s) for s in self.seqs]

        # Pre-tokenize each sequence individually
        bc = alphabet.get_batch_converter()
        self.tokens = []
        for slug, seq in zip(self.slugs, self.seqs):
            _, _, tok = bc([(slug, seq)])
            self.tokens.append(tok.squeeze(0))  # (seq_len + 2,)

        # Chromophore positions (0-indexed, -1 if unavailable)
        self.chrom_pos = torch.full((len(df), 3), -1, dtype=torch.long)
        for i, (_, row) in enumerate(df.iterrows()):
            if row.get("status") == "ok":
                self.chrom_pos[i, 0] = int(row["chrom_pos0"])
                self.chrom_pos[i, 1] = int(row["chrom_pos1"])
                self.chrom_pos[i, 2] = int(row["chrom_pos2"])

        # Z-score targets (fit on train set via target_stats)
        self.target_stats = target_stats if target_stats is not None else {}
        self.targets = {}
        for t in TARGETS:
            vals = df[t].values.astype(np.float64)
            if t not in self.target_stats:
                valid = ~np.isnan(vals)
                mu = float(np.nanmean(vals)) if valid.any() else 0.0
                sd = float(max(np.nanstd(vals), 1e-8)) if valid.any() else 1.0
                self.target_stats[t] = (mu, sd)
            mu, sd = self.target_stats[t]
            self.targets[t] = torch.tensor((vals - mu) / sd, dtype=torch.float32)

    def __len__(self):
        return len(self.slugs)

    def __getitem__(self, idx):
        item = {
            "tokens": self.tokens[idx],
            "seq_len": self.seq_lens[idx],
            "chrom_pos": self.chrom_pos[idx],
        }
        for t in TARGETS:
            item[f"target_{t}"] = self.targets[t][idx]
        return item


def collate_fn(batch):
    """Pad tokens to max length in batch."""
    max_len = max(b["tokens"].shape[0] for b in batch)
    B = len(batch)

    tokens    = torch.full((B, max_len), 1, dtype=torch.long)  # 1 = <pad>
    seq_lens  = torch.zeros(B, dtype=torch.long)
    chrom_pos = torch.full((B, 3), -1, dtype=torch.long)
    targets   = {t: torch.full((B,), float("nan")) for t in TARGETS}

    for i, b in enumerate(batch):
        L = b["tokens"].shape[0]
        tokens[i, :L] = b["tokens"]
        seq_lens[i] = b["seq_len"]
        chrom_pos[i] = b["chrom_pos"]
        for t in TARGETS:
            targets[t][i] = b[f"target_{t}"]

    return tokens, seq_lens, chrom_pos, targets


print("Dataset & collate ready.")

In [ ]:
# ── Cell 7: Training function ─────────────────────────────────────────────────

def reset_trainable_weights(model):
    """Re-initialize all trainable weights (LoRA + pooling + heads) for a new fold."""
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if "lora_A" in name:
            nn.init.normal_(param, std=0.01)
        elif "lora_B" in name:
            nn.init.zeros_(param)

    for module in [model.pool, model.heads]:
        for m in module.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    model.pool.chrom_bias.data.fill_(3.0)  # ↑ from 1.0


def train_one_fold(fold_idx, train_df, val_df, model, alphabet):
    """Train for one fold. Returns (best_val_loss, history) or (None, None) if resumed."""
    fold_dir = OUT_DIR / f"fold_{fold_idx}"
    fold_dir.mkdir(exist_ok=True)
    ckpt_path = fold_dir / "best.pt"

    if ckpt_path.exists():
        print(f"  Fold {fold_idx}: checkpoint exists — skipping")
        return None, None

    # ── Target stats from train set only (prevent leakage) ────────────────
    target_stats = {}
    for t in TARGETS:
        vals = train_df[t].dropna().values
        if len(vals) > 0:
            target_stats[t] = (float(np.mean(vals)), float(max(np.std(vals), 1e-8)))
        else:
            target_stats[t] = (0.0, 1.0)

    train_ds = FPDataset(train_df, alphabet, target_stats)
    val_ds   = FPDataset(val_df, alphabet, target_stats)

    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, drop_last=True, num_workers=2)
    val_dl   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=2)

    # ── Reset weights for this fold ───────────────────────────────────────
    reset_trainable_weights(model)

    # ── Optimizer with differential LR ────────────────────────────────────
    lora_ps  = [p for n, p in model.named_parameters() if p.requires_grad and "lora_" in n]
    other_ps = [p for n, p in model.named_parameters() if p.requires_grad and "lora_" not in n]

    optimizer = torch.optim.AdamW([
        {"params": lora_ps, "lr": LR_LORA},
        {"params": other_ps, "lr": LR_HEAD},
    ], weight_decay=WEIGHT_DECAY)

    # Cosine annealing after warmup
    cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=MAX_EPOCHS - WARMUP_EPOCHS, eta_min=1e-6,
    )
    # Linear warmup for first WARMUP_EPOCHS
    warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.01, end_factor=1.0, total_iters=WARMUP_EPOCHS,
    )
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[warmup_scheduler, cosine_scheduler],
        milestones=[WARMUP_EPOCHS],
    )

    scaler = GradScaler()

    best_val_loss = float("inf")
    patience_ctr = 0
    history = []

    for epoch in range(MAX_EPOCHS):
        # ── Train ─────────────────────────────────────────────────────────
        model.train()
        model.esm.eval()  # keep frozen layers in eval (no dropout)
        train_losses = []
        optimizer.zero_grad()

        for step, (tokens, seq_lens, chrom_pos, tgts) in enumerate(train_dl):
            tokens   = tokens.to(DEVICE)
            seq_lens = seq_lens.to(DEVICE)
            chrom_pos = chrom_pos.to(DEVICE)
            tgts = {t: v.to(DEVICE) for t, v in tgts.items()}

            with autocast():
                preds, _, _ = model(tokens, seq_lens, chrom_pos)
                loss = masked_huber_loss(preds, tgts) / ACCUM_STEPS

            scaler.scale(loss).backward()
            train_losses.append(loss.item() * ACCUM_STEPS)

            if (step + 1) % ACCUM_STEPS == 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                torch.cuda.empty_cache()

        # Flush remaining grads
        if (step + 1) % ACCUM_STEPS != 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        # ── Validate ──────────────────────────────────────────────────────
        model.eval()
        val_losses = []
        with torch.no_grad(), autocast():
            for tokens, seq_lens, chrom_pos, tgts in val_dl:
                tokens   = tokens.to(DEVICE)
                seq_lens = seq_lens.to(DEVICE)
                chrom_pos = chrom_pos.to(DEVICE)
                tgts = {t: v.to(DEVICE) for t, v in tgts.items()}

                preds, _, _ = model(tokens, seq_lens, chrom_pos)
                val_losses.append(masked_huber_loss(preds, tgts).item())

        t_loss = np.mean(train_losses)
        v_loss = np.mean(val_losses) if val_losses else float("inf")
        lr_now = scheduler.get_last_lr()[0]

        history.append({"epoch": epoch, "train_loss": t_loss, "val_loss": v_loss, "lr": lr_now})

        if epoch % 10 == 0 or v_loss < best_val_loss:
            print(f"    Ep {epoch+1:3d}  train={t_loss:.4f}  val={v_loss:.4f}  lr={lr_now:.2e}")

        # ── Early stopping + checkpoint ───────────────────────────────────
        if v_loss < best_val_loss:
            best_val_loss = v_loss
            patience_ctr = 0
            torch.save({
                "trainable_state": {
                    k: v.cpu() for k, v in model.state_dict().items()
                    if any(x in k for x in ["lora_", "pool.", "heads."])
                },
                "target_stats": target_stats,
                "epoch": epoch,
                "val_loss": v_loss,
            }, ckpt_path)
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"    Early stopping at epoch {epoch + 1}")
                break

        scheduler.step()

    pd.DataFrame(history).to_csv(fold_dir / "history.csv", index=False)
    return best_val_loss, history


print("Training function ready.")

In [ ]:
# ── Cell 8: Run 20-fold CV ────────────────────────────────────────────────────
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
all_indices = np.arange(len(meta))

print(f"Starting {N_FOLDS}-fold CV on {len(meta)} samples...\n")
t_total = time.time()
fold_results = []

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(all_indices)):
    print(f"── Fold {fold_idx} ({len(train_idx)} train / {len(val_idx)} val) ──")
    t_fold = time.time()

    train_df = meta.iloc[train_idx].reset_index(drop=True)
    val_df   = meta.iloc[val_idx].reset_index(drop=True)

    best_loss, history = train_one_fold(fold_idx, train_df, val_df, model, alphabet)

    if best_loss is not None:
        elapsed = time.time() - t_fold
        print(f"  Best val loss: {best_loss:.4f} ({elapsed / 60:.1f} min)\n")
        fold_results.append({
            "fold": fold_idx, "best_val_loss": best_loss, "time_min": elapsed / 60,
        })
    else:
        print()

total_time = time.time() - t_total
print(f"\nAll folds complete in {total_time / 3600:.1f} hours")

if fold_results:
    rdf = pd.DataFrame(fold_results)
    rdf.to_csv(OUT_DIR / "fold_training_results.csv", index=False)
    print(f"Mean val loss: {rdf['best_val_loss'].mean():.4f} ± {rdf['best_val_loss'].std():.4f}")

In [ ]:
# ── Cell 9: Evaluation — predict on val sets & compute metrics ────────────────
from scipy.stats import pearsonr, spearmanr

print("Evaluating all folds...\n")

CLAMP_RANGES = {
    "ex_max": (300, 800), "em_max": (300, 800),
    "qy": (0.0, 1.0), "ext_coeff": (0, 300000), "pka": (0, 14),
}

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
all_preds = {t: np.full(len(meta), np.nan) for t in TARGETS}

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(np.arange(len(meta)))):
    fold_dir  = OUT_DIR / f"fold_{fold_idx}"
    ckpt_path = fold_dir / "best.pt"

    if not ckpt_path.exists():
        print(f"  Fold {fold_idx}: no checkpoint — skipping")
        continue

    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["trainable_state"], strict=False)
    target_stats = ckpt["target_stats"]

    val_df = meta.iloc[val_idx].reset_index(drop=True)
    val_ds = FPDataset(val_df, alphabet, target_stats)
    val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        collate_fn=collate_fn, num_workers=0)

    model.eval()
    fold_preds = {t: [] for t in TARGETS}

    with torch.no_grad(), autocast():
        for tokens, seq_lens, chrom_pos, _ in val_dl:
            tokens    = tokens.to(DEVICE)
            seq_lens  = seq_lens.to(DEVICE)
            chrom_pos = chrom_pos.to(DEVICE)

            preds, _, _ = model(tokens, seq_lens, chrom_pos)
            for t in TARGETS:
                mu, sd = target_stats[t]
                fold_preds[t].append(preds[t].cpu().numpy() * sd + mu)

    for t in TARGETS:
        pred_vals = np.concatenate(fold_preds[t])
        lo, hi = CLAMP_RANGES[t]
        pred_vals = np.clip(pred_vals, lo, hi)
        for i, idx in enumerate(val_idx):
            if i < len(pred_vals):
                all_preds[t][idx] = pred_vals[i]

    print(f"  Fold {fold_idx}: predicted {len(val_idx)} samples")

# ── Compute metrics ───────────────────────────────────────────────────────────
print(f"\n{'=' * 75}")
print(f"  LoRA ESM2-650M + ChromophorePool — {N_FOLDS}-Fold CV Results")
print(f"{'=' * 75}\n")

metrics = {}
for t in TARGETS:
    pred = all_preds[t]
    true = meta[t].values
    valid = ~np.isnan(true) & ~np.isnan(pred) & np.isfinite(pred)
    if valid.sum() == 0:
        print(f"  {t:12s}  ** skipped — no finite predictions **")
        continue

    mae  = mean_absolute_error(true[valid], pred[valid])
    rmse = np.sqrt(mean_squared_error(true[valid], pred[valid]))
    r, _  = pearsonr(true[valid], pred[valid])
    rho, _ = spearmanr(true[valid], pred[valid])
    r2   = 1 - np.sum((true[valid] - pred[valid]) ** 2) / np.sum((true[valid] - true[valid].mean()) ** 2)

    metrics[t] = {"MAE": float(mae), "RMSE": float(rmse), "R": float(r),
                  "Rho": float(rho), "R2": float(r2), "N": int(valid.sum())}

    unit = " nm" if t in ("ex_max", "em_max") else ""
    print(f"  {t:12s}  N={valid.sum():3d}  MAE={mae:7.2f}{unit}  RMSE={rmse:7.2f}{unit}  R={r:.3f}  ρ={rho:.3f}  R²={r2:.3f}")

# ── Comparison vs FPredX baseline ─────────────────────────────────────────────
fpredx = {"ex_max": 14.76, "em_max": 9.38, "qy": 0.106, "ext_coeff": 17837, "pka": 0.696}

print(f"\n── vs FPredX (improved) baseline ──")
for t, baseline in fpredx.items():
    if t in metrics:
        ours = metrics[t]["MAE"]
        delta = ours - baseline
        tag = "BETTER" if delta < 0 else "worse"
        print(f"  {t:12s}  Ours={ours:.2f}  Baseline={baseline}  ({tag}, Δ={delta:+.2f})")

with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# ── Scatter plots ─────────────────────────────────────────────────────────────
n_plots = len(metrics)
fig, axes = plt.subplots(1, n_plots, figsize=(4 * n_plots, 4))
if n_plots == 1:
    axes = [axes]

for ax, t in zip(axes, metrics):
    pred = all_preds[t]
    true = meta[t].values
    v = ~np.isnan(true) & ~np.isnan(pred) & np.isfinite(pred)
    ax.scatter(true[v], pred[v], alpha=0.4, s=12)
    lo, hi = true[v].min(), true[v].max()
    ax.plot([lo, hi], [lo, hi], "r--", lw=1)
    ax.set_xlabel("True")
    ax.set_ylabel("Predicted")
    ax.set_title(f"{t}\nMAE={metrics[t]['MAE']:.2f}  R²={metrics[t]['R2']:.3f}  ρ={metrics[t]['Rho']:.3f}")

plt.tight_layout()
plt.savefig(str(OUT_DIR / "scatter_plots.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"\nSaved metrics → {OUT_DIR / 'metrics.json'}")
print(f"Saved plots  → {OUT_DIR / 'scatter_plots.png'}")

In [ ]:
# ── Cell 10: Extract embeddings for future late fusion ────────────────────────
# For each fold's best model, extract pooled 1280-dim embeddings for ALL FPs.
# These can later be combined with structural features → XGBoost.

print("Extracting LoRA-pooled embeddings for all folds...\n")

# Build a DataLoader over all samples (dummy targets, we just need embeddings)
dummy_stats = {t: (0.0, 1.0) for t in TARGETS}
full_ds = FPDataset(meta, alphabet, dummy_stats)
full_dl = DataLoader(full_ds, batch_size=BATCH_SIZE, shuffle=False,
                     collate_fn=collate_fn, num_workers=0)

all_fold_embs = np.zeros((N_FOLDS, len(meta), EMBED_DIM), dtype=np.float32)

for fold_idx in range(N_FOLDS):
    ckpt_path = OUT_DIR / f"fold_{fold_idx}" / "best.pt"
    if not ckpt_path.exists():
        print(f"  Fold {fold_idx}: no checkpoint — skipping")
        continue

    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["trainable_state"], strict=False)
    model.eval()

    embs = []
    with torch.no_grad(), autocast():
        for tokens, seq_lens, chrom_pos, _ in full_dl:
            tokens    = tokens.to(DEVICE)
            seq_lens  = seq_lens.to(DEVICE)
            chrom_pos = chrom_pos.to(DEVICE)
            _, _, pooled = model(tokens, seq_lens, chrom_pos)
            embs.append(pooled.cpu().numpy())

    fold_embs = np.concatenate(embs, axis=0)[:len(meta)]
    all_fold_embs[fold_idx] = fold_embs
    print(f"  Fold {fold_idx}: {fold_embs.shape}")

out_path = OUT_DIR / "lora_embeddings_all_folds.npz"
np.savez_compressed(
    str(out_path),
    embeddings=all_fold_embs,
    slugs=np.array(meta["slug"].tolist()),
)

size_gb = all_fold_embs.nbytes / 1e9
print(f"\nSaved: {out_path}")
print(f"Shape: {all_fold_embs.shape}  ({size_gb:.2f} GB)")
print(f"\nUse these embeddings with train_late_fusion.py for structural integration.")

In [ ]:
# ── Cell 11: Late Fusion — XGBoost on one-hot + LoRA embeddings ───────────────
# Combines alignment-based one-hot features (what FPredX uses) with the
# LoRA-pooled embeddings from cell 10. XGBoost acts as the final predictor.
# Optuna tunes hyperparameters for the fused feature set.
!pip install -q optuna xgboost
from xgboost import XGBRegressor
from collections import Counter
from scipy.stats import pearsonr, spearmanr
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── 1. Build one-hot features from alignment ─────────────────────────────────
ALN_PATH = DRIVE_DIR / "sequence" / "fp_sequences_aligned.fasta"

def load_alignment(fasta_path):
    """Parse aligned FASTA into {slug: aligned_sequence}."""
    aligned = {}
    current_slug = None
    current_seq = []
    with open(fasta_path) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if current_slug:
                    aligned[current_slug] = "".join(current_seq)
                current_slug = line[1:].split()[0]
                current_seq = []
            else:
                current_seq.append(line)
    if current_slug:
        aligned[current_slug] = "".join(current_seq)
    return aligned

def build_onehot_features(slugs, aligned, min_freq=0.02):
    """Build alignment-based one-hot features (FPredX style)."""
    aln_len = len(next(iter(aligned.values())))
    n_seqs = len(slugs)

    position_chars = []
    for pos in range(aln_len):
        chars = Counter()
        for slug in slugs:
            if slug in aligned:
                chars[aligned[slug][pos]] += 1
        for char, count in chars.items():
            if count / n_seqs >= min_freq and char != "-":
                position_chars.append((pos, char))

    X = np.zeros((n_seqs, len(position_chars)), dtype=np.float32)
    for i, slug in enumerate(slugs):
        if slug not in aligned:
            continue
        seq = aligned[slug]
        for j, (pos, char) in enumerate(position_chars):
            if seq[pos] == char:
                X[i, j] = 1.0

    var = X.var(axis=0)
    keep = var > 0
    X = X[:, keep]
    kept_pairs = [pc for pc, k in zip(position_chars, keep) if k]
    return X, kept_pairs

print("Loading alignment...")
aligned = load_alignment(ALN_PATH)
slugs = meta["slug"].tolist()
X_onehot, feat_pairs = build_onehot_features(slugs, aligned)
print(f"One-hot features: {X_onehot.shape[1]} (from {len(aligned)} aligned sequences)")

# ── 2. Load LoRA embeddings ──────────────────────────────────────────────────
emb_path = OUT_DIR / "lora_embeddings_all_folds.npz"
emb_data = np.load(str(emb_path))
all_fold_embs = emb_data["embeddings"]  # (20, 927, 1280)
print(f"LoRA embeddings: {all_fold_embs.shape}")

# ── 3. Optuna tuning for fused features ──────────────────────────────────────
OPTUNA_TRIALS = 30
OPTUNA_CV = 3

def tune_xgb_for_target(target_name, X_full, y_full, n_trials=OPTUNA_TRIALS):
    """Tune XGBoost hyperparams on the fused feature set using inner CV."""
    valid_mask = ~np.isnan(y_full)
    X_v = X_full[valid_mask]
    y_v = y_full[valid_mask]

    if len(y_v) < 30:
        print(f"  {target_name}: too few samples ({len(y_v)}) — using defaults")
        return dict(n_estimators=500, max_depth=7, learning_rate=0.03,
                    subsample=0.8, colsample_bytree=0.5,
                    reg_alpha=1.0, reg_lambda=1.0, min_child_weight=1)

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 300, 1500),
            "max_depth": trial.suggest_int("max_depth", 3, 9),
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.15, log=True),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.9),
            "reg_alpha": trial.suggest_float("reg_alpha", 0.01, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.01, 10.0, log=True),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        }

        kf_inner = KFold(n_splits=OPTUNA_CV, shuffle=True, random_state=RANDOM_SEED)
        maes = []
        for tr_idx, va_idx in kf_inner.split(X_v):
            mdl = XGBRegressor(**params, random_state=RANDOM_SEED, n_jobs=-1,
                               tree_method="hist")
            mdl.fit(X_v[tr_idx], y_v[tr_idx], verbose=False)
            pred = mdl.predict(X_v[va_idx])
            maes.append(mean_absolute_error(y_v[va_idx], pred))
        return np.mean(maes)

    study = optuna.create_study(direction="minimize",
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best = study.best_params
    print(f"  {target_name}: best MAE={study.best_value:.4f} "
          f"(n_est={best['n_estimators']}, depth={best['max_depth']}, "
          f"lr={best['learning_rate']:.4f}, colsample={best['colsample_bytree']:.3f})")
    return best

# Tune per target using fold-0 embeddings as representative
print(f"\nTuning XGBoost for fused features ({OPTUNA_TRIALS} trials × {OPTUNA_CV}-fold)...\n")
X_fused_tune = np.hstack([X_onehot, all_fold_embs[0]])
tuned_params = {}
for t in TARGETS:
    tuned_params[t] = tune_xgb_for_target(t, X_fused_tune, meta[t].values)

# ── 4. 20-fold CV with tuned fusion ──────────────────────────────────────────
print(f"\nRunning {N_FOLDS}-fold XGBoost late fusion (tuned)...\n")

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
fusion_preds = {t: np.full(len(meta), np.nan) for t in TARGETS}

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(np.arange(len(meta)))):
    fold_embs = all_fold_embs[fold_idx]
    X_fused = np.hstack([X_onehot, fold_embs])

    for t in TARGETS:
        y = meta[t].values
        train_valid = ~np.isnan(y[train_idx])
        val_valid   = ~np.isnan(y[val_idx])

        if train_valid.sum() < 10 or val_valid.sum() == 0:
            continue

        X_tr = X_fused[train_idx[train_valid]]
        y_tr = y[train_idx[train_valid]]
        X_va = X_fused[val_idx[val_valid]]

        xgb = XGBRegressor(
            **tuned_params[t], random_state=RANDOM_SEED, n_jobs=-1, tree_method="hist",
        )
        xgb.fit(X_tr, y_tr, verbose=False)
        pred_va = xgb.predict(X_va)

        lo, hi = CLAMP_RANGES[t]
        pred_va = np.clip(pred_va, lo, hi)

        for i, idx in enumerate(val_idx[val_valid]):
            fusion_preds[t][idx] = pred_va[i]

    if fold_idx % 5 == 0:
        print(f"  Fold {fold_idx} done")

print(f"  All {N_FOLDS} folds done")

# ── 5. Compute metrics ───────────────────────────────────────────────────────
print(f"\n{'=' * 75}")
print(f"  Late Fusion (One-Hot + LoRA → Tuned XGBoost) — {N_FOLDS}-Fold CV")
print(f"{'=' * 75}\n")

fusion_metrics = {}
for t in TARGETS:
    pred = fusion_preds[t]
    true = meta[t].values
    valid = ~np.isnan(true) & ~np.isnan(pred) & np.isfinite(pred)
    if valid.sum() == 0:
        print(f"  {t:12s}  ** skipped **")
        continue

    mae  = mean_absolute_error(true[valid], pred[valid])
    rmse = np.sqrt(mean_squared_error(true[valid], pred[valid]))
    r, _  = pearsonr(true[valid], pred[valid])
    rho, _ = spearmanr(true[valid], pred[valid])
    r2   = 1 - np.sum((true[valid] - pred[valid]) ** 2) / np.sum((true[valid] - true[valid].mean()) ** 2)

    fusion_metrics[t] = {"MAE": float(mae), "RMSE": float(rmse), "R": float(r),
                         "Rho": float(rho), "R2": float(r2), "N": int(valid.sum())}

    unit = " nm" if t in ("ex_max", "em_max") else ""
    print(f"  {t:12s}  N={valid.sum():3d}  MAE={mae:7.2f}{unit}  RMSE={rmse:7.2f}{unit}  R={r:.3f}  ρ={rho:.3f}  R²={r2:.3f}")

# ── 6. Comparison table ──────────────────────────────────────────────────────
fpredx = {"ex_max": 14.76, "em_max": 9.38, "qy": 0.106, "ext_coeff": 17837, "pka": 0.696}

print(f"\n── Comparison ──")
print(f"  {'Target':12s}  {'FPredX':>10s}  {'LoRA only':>10s}  {'Fusion':>10s}  {'vs FPredX':>10s}")
print(f"  {'─'*12}  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*10}")
for t in TARGETS:
    base = fpredx.get(t, None)
    lora = metrics.get(t, {}).get("MAE", None)
    fuse = fusion_metrics.get(t, {}).get("MAE", None)
    if base and fuse:
        delta = fuse - base
        tag = "BETTER" if delta < 0 else "worse"
        print(f"  {t:12s}  {base:>10.2f}  {lora:>10.2f}  {fuse:>10.2f}  {delta:>+9.2f} ({tag})")

with open(OUT_DIR / "fusion_metrics.json", "w") as f:
    json.dump(fusion_metrics, f, indent=2)
with open(OUT_DIR / "fusion_tuned_params.json", "w") as f:
    json.dump(tuned_params, f, indent=2)

# ── 7. Scatter plots ─────────────────────────────────────────────────────────
n_plots = len(fusion_metrics)
fig, axes = plt.subplots(1, n_plots, figsize=(4 * n_plots, 4))
if n_plots == 1:
    axes = [axes]

for ax, t in zip(axes, fusion_metrics):
    pred = fusion_preds[t]
    true = meta[t].values
    v = ~np.isnan(true) & ~np.isnan(pred) & np.isfinite(pred)
    ax.scatter(true[v], pred[v], alpha=0.4, s=12)
    lo, hi = true[v].min(), true[v].max()
    ax.plot([lo, hi], [lo, hi], "r--", lw=1)
    ax.set_xlabel("True")
    ax.set_ylabel("Predicted")
    ax.set_title(f"{t}\nMAE={fusion_metrics[t]['MAE']:.2f}  R²={fusion_metrics[t]['R2']:.3f}  ρ={fusion_metrics[t]['Rho']:.3f}")

plt.tight_layout()
plt.savefig(str(OUT_DIR / "fusion_scatter_plots.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"\nSaved → {OUT_DIR / 'fusion_metrics.json'}")
print(f"Saved → {OUT_DIR / 'fusion_tuned_params.json'}")
print(f"Saved → {OUT_DIR / 'fusion_scatter_plots.png'}")

## Cell 12 — Fusion + Structural Features (H100 GPU XGBoost)

Stacks the cell-11 features (one-hot + LoRA) with two additional structural matrices produced by `model/LoRA_ESM2_Structure/build_pocket3d_features.ipynb`:

- **Hand-crafted CA features** (`structural_features.npz`, ~55 dims): chromophore triad geometry + 6/8/10 Å shell composition + global Rg / contact order
- **ESM-IF1 pooled embeddings** (`esmif_embeddings.npz`, 2048 dims): structure-aware GVP-GNN embeddings, pooled as `[mean | max | chrom±5 mean | chrom±10 mean]`
- `has_structure` indicator so XGBoost can route the 14 GFPs without a structure separately

Switches XGBoost to `device="cuda"` (H100), bumps Optuna to **200 trials** with `MedianPruner`, and averages predictions across **3 random seeds** in the outer 20-fold CV. Goal: beat FPredX on `ex_max` (13.53) and `em_max` (9.06).

In [ ]:
# ── Cell 12: GPU XGBoost fusion with structural features ─────────────────────
# Mirrors cell 11 but adds two structural feature matrices and runs every
# XGBoost fit/predict on the H100 (device="cuda"). Multi-seed outer CV +
# Optuna 200 trials with MedianPruner exploit the H100 budget.
import xgboost as xgb
print("xgboost version:", xgb.__version__)
assert tuple(int(x) for x in xgb.__version__.split(".")[:2]) >= (2, 0), \
    "xgboost >= 2.0 required for device='cuda'. Run: !pip install -q -U xgboost"

!nvidia-smi --query-gpu=name,utilization.gpu,memory.used,memory.total --format=csv

# ── 1. Load structural feature matrices (built in build_struct_features.ipynb)
STRUCT_DIR = Path("/content/drive/MyDrive/FluorCode/struct_features")
hand  = np.load(STRUCT_DIR / "structural_features.npz", allow_pickle=True)
esmif = np.load(STRUCT_DIR / "esmif_embeddings.npz",   allow_pickle=True)

slug2hand  = {s: hand["features"][i]  for i, s in enumerate(hand["slugs"])}
slug2esmif = {s: esmif["features"][i] for i, s in enumerate(esmif["slugs"])}
slug2has   = {s: int(hand["has_structure"][i]) for i, s in enumerate(hand["slugs"])}
D_hand  = hand["features"].shape[1]
D_esmif = esmif["features"].shape[1]

X_hand  = np.stack([slug2hand .get(s, np.zeros(D_hand,  dtype=np.float32)) for s in slugs]).astype(np.float32)
X_esmif = np.stack([slug2esmif.get(s, np.zeros(D_esmif, dtype=np.float32)) for s in slugs]).astype(np.float32)
has_struct = np.array([slug2has.get(s, 0) for s in slugs], dtype=np.float32)[:, None]

# Sanitize: replace any NaN/Inf with zeros (XGBoost handles missing but be explicit)
X_hand[~np.isfinite(X_hand)]   = 0.0
X_esmif[~np.isfinite(X_esmif)] = 0.0

print(f"Hand-crafted struct features: {X_hand.shape}  ({int(has_struct.sum())}/{len(slugs)} with structure)")
print(f"ESM-IF1 struct embeddings:    {X_esmif.shape}")
print(f"One-hot:                      {X_onehot.shape}")
print(f"LoRA per-fold embeddings:     {all_fold_embs.shape}")
total_dim = X_onehot.shape[1] + all_fold_embs.shape[2] + D_hand + D_esmif + 1
print(f"Total feature dim per FP:     {total_dim}")

# ── 2. Optuna tuning on fused (one-hot + LoRA fold-0 + struct) features ──────
OPTUNA_TRIALS_S = 200
OPTUNA_CV_S    = 3
SEEDS          = (42, 1337, 2024)

def tune_xgb_struct(target_name, X_full, y_full, n_trials=OPTUNA_TRIALS_S):
    valid = ~np.isnan(y_full)
    X_v, y_v = X_full[valid], y_full[valid]
    if len(y_v) < 30:
        print(f"  {target_name}: too few samples ({len(y_v)}) — defaults")
        return dict(n_estimators=500, max_depth=7, learning_rate=0.03,
                    subsample=0.8, colsample_bytree=0.5,
                    reg_alpha=1.0, reg_lambda=1.0, min_child_weight=1)

    def objective(trial):
        params = {
            "n_estimators":     trial.suggest_int("n_estimators", 300, 2000),
            "max_depth":        trial.suggest_int("max_depth", 3, 10),
            "learning_rate":    trial.suggest_float("learning_rate", 0.005, 0.15, log=True),
            "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.9),
            "reg_alpha":        trial.suggest_float("reg_alpha",  0.01, 10.0, log=True),
            "reg_lambda":       trial.suggest_float("reg_lambda", 0.01, 10.0, log=True),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        }
        kf_in = KFold(n_splits=OPTUNA_CV_S, shuffle=True, random_state=RANDOM_SEED)
        maes = []
        for k, (tr, va) in enumerate(kf_in.split(X_v)):
            mdl = XGBRegressor(**params, random_state=RANDOM_SEED, n_jobs=1,
                               tree_method="hist", device="cuda")
            mdl.fit(X_v[tr], y_v[tr], verbose=False)
            pred = mdl.predict(X_v[va])
            mae = mean_absolute_error(y_v[va], pred)
            trial.report(mae, k)
            if trial.should_prune():
                raise optuna.TrialPruned()
            maes.append(mae)
        return float(np.mean(maes))

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=1),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best = study.best_params
    print(f"  {target_name}: best MAE={study.best_value:.4f}  "
          f"n_est={best['n_estimators']}  depth={best['max_depth']}  "
          f"lr={best['learning_rate']:.4f}")
    return best

print(f"\nTuning (200 trials × 3-fold inner CV, GPU XGBoost)…\n")
X_tune = np.hstack([X_onehot, all_fold_embs[0], X_hand, X_esmif, has_struct])
print(f"Tuning matrix: {X_tune.shape}")
t0 = time.time()
tuned_params_s = {t: tune_xgb_struct(t, X_tune, meta[t].values) for t in TARGETS}
print(f"Tuning wall-time: {(time.time() - t0):.1f}s")

!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv

# ── 3. Multi-seed × 20-fold outer CV ────────────────────────────────────────
print(f"\nRunning {N_FOLDS}-fold × {len(SEEDS)}-seed outer CV (GPU XGBoost)…\n")
# accumulators: sum of predictions across seeds, then divide by seed count seen per index
pred_sum   = {t: np.zeros(len(meta), dtype=np.float64) for t in TARGETS}
pred_count = {t: np.zeros(len(meta), dtype=np.int32)   for t in TARGETS}

t0 = time.time()
for seed in SEEDS:
    print(f"── seed {seed} ──")
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(np.arange(len(meta)))):
        # Reuse the same per-fold LoRA embedding bank produced in cell 10.
        # (Embeddings were trained with seed=42 KFold; mixing seeds here is a
        # mild approximation but the LoRA repr varies smoothly with fold.)
        fold_embs = all_fold_embs[fold_idx]
        X_full = np.hstack([X_onehot, fold_embs, X_hand, X_esmif, has_struct])

        for t in TARGETS:
            y = meta[t].values
            tr_ok = ~np.isnan(y[train_idx])
            va_ok = ~np.isnan(y[val_idx])
            if tr_ok.sum() < 10 or va_ok.sum() == 0:
                continue
            X_tr = X_full[train_idx[tr_ok]]
            y_tr = y[train_idx[tr_ok]]
            X_va = X_full[val_idx[va_ok]]
            mdl = XGBRegressor(
                **tuned_params_s[t],
                random_state=seed, n_jobs=1,
                tree_method="hist", device="cuda",
            )
            mdl.fit(X_tr, y_tr, verbose=False)
            pred_va = np.clip(mdl.predict(X_va), *CLAMP_RANGES[t])
            for i, idx in enumerate(val_idx[va_ok]):
                pred_sum[t][idx]   += pred_va[i]
                pred_count[t][idx] += 1
        if fold_idx % 5 == 0:
            print(f"  seed {seed} fold {fold_idx} done  ({(time.time()-t0):.1f}s)")
print(f"\nOuter CV wall-time: {(time.time()-t0):.1f}s")

!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv

# Average predictions across seeds (per-index, since CV folds differ per seed)
struct_preds = {}
for t in TARGETS:
    avg = np.full(len(meta), np.nan, dtype=np.float64)
    nz  = pred_count[t] > 0
    avg[nz] = pred_sum[t][nz] / pred_count[t][nz]
    struct_preds[t] = avg

# ── 4. Compute metrics ──────────────────────────────────────────────────────
print(f"\n{'=' * 78}")
print(f"  Late Fusion + Struct (One-Hot + LoRA + Hand + ESM-IF1) — {N_FOLDS}-fold × {len(SEEDS)}-seed")
print(f"{'=' * 78}\n")

struct_metrics = {}
for t in TARGETS:
    pred = struct_preds[t]
    true = meta[t].values
    valid = ~np.isnan(true) & ~np.isnan(pred) & np.isfinite(pred)
    if valid.sum() == 0:
        print(f"  {t:12s}  ** skipped **"); continue
    mae  = mean_absolute_error(true[valid], pred[valid])
    rmse = np.sqrt(mean_squared_error(true[valid], pred[valid]))
    r,   _ = pearsonr(true[valid],  pred[valid])
    rho, _ = spearmanr(true[valid], pred[valid])
    r2 = 1 - np.sum((true[valid] - pred[valid]) ** 2) / np.sum((true[valid] - true[valid].mean()) ** 2)
    struct_metrics[t] = {"MAE": float(mae), "RMSE": float(rmse), "R": float(r),
                         "Rho": float(rho), "R2": float(r2), "N": int(valid.sum())}
    unit = " nm" if t in ("ex_max", "em_max") else ""
    print(f"  {t:12s}  N={valid.sum():3d}  MAE={mae:7.3f}{unit}  RMSE={rmse:7.3f}{unit}  R={r:.3f}  ρ={rho:.3f}  R²={r2:.3f}")

# ── 5. Comparison vs FPredX (GFP-only published numbers) ────────────────────
fpredx_gfp = {"ex_max": 13.53, "em_max": 9.06, "qy": 0.111, "ext_coeff": 17994, "pka": 0.72}

print(f"\n── Comparison (GFP family, {N_FOLDS}-fold CV) ──")
print(f"  {'Target':12s}  {'FPredX':>10s}  {'Fusion (cell11)':>16s}  {'+Struct (cell12)':>18s}  {'Δ vs FPredX':>14s}")
print(f"  {'─'*12}  {'─'*10}  {'─'*16}  {'─'*18}  {'─'*14}")
for t in TARGETS:
    base = fpredx_gfp.get(t)
    fuse = fusion_metrics.get(t, {}).get("MAE")
    full = struct_metrics.get(t, {}).get("MAE")
    if base is None or full is None: continue
    delta = full - base
    tag = "BETTER" if delta < 0 else "worse"
    fuse_str = f"{fuse:>16.3f}" if fuse is not None else f"{'—':>16s}"
    print(f"  {t:12s}  {base:>10.3f}  {fuse_str}  {full:>18.3f}  {delta:>+11.3f} ({tag})")

# ── 6. Save artifacts ───────────────────────────────────────────────────────
with open(OUT_DIR / "fusion_struct_metrics.json", "w") as f:
    json.dump(struct_metrics, f, indent=2)
with open(OUT_DIR / "fusion_struct_tuned_params.json", "w") as f:
    json.dump(tuned_params_s, f, indent=2)

n_plots = len(struct_metrics)
fig, axes = plt.subplots(1, n_plots, figsize=(4 * n_plots, 4))
if n_plots == 1: axes = [axes]
for ax, t in zip(axes, struct_metrics):
    pred = struct_preds[t]; true = meta[t].values
    v = ~np.isnan(true) & ~np.isnan(pred) & np.isfinite(pred)
    ax.scatter(true[v], pred[v], alpha=0.4, s=12)
    lo, hi = true[v].min(), true[v].max()
    ax.plot([lo, hi], [lo, hi], "r--", lw=1)
    ax.set_xlabel("True"); ax.set_ylabel("Predicted")
    m = struct_metrics[t]
    ax.set_title(f"{t}\nMAE={m['MAE']:.2f}  R²={m['R2']:.3f}  ρ={m['Rho']:.3f}")
plt.tight_layout()
plt.savefig(str(OUT_DIR / "scatter_struct.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"\nSaved → {OUT_DIR / 'fusion_struct_metrics.json'}")
print(f"Saved → {OUT_DIR / 'fusion_struct_tuned_params.json'}")
print(f"Saved → {OUT_DIR / 'scatter_struct.png'}")